In [0]:
import data.utilities.common as Commonlib

import pyspark.sql.functions as F
import yaml

from data.utilities.medallion import Medallion
from data.utilities.medallion_layer import MedallionLayer as ML
from data.utilities.medallion_table_utility import MedallionTableUtility as MTU
from data.utilities.publishers import SnowflakeExternalTable
from pyspark.sql.types import TimestampType
from pyspark.sql.window import Window

In [0]:
CONFIG_BASE_PATH = "configuration/"

config_selection_path = f"{CONFIG_BASE_PATH}"
config_entries = Commonlib.list_files(config_selection_path, max_depth=1)
dbutils.widgets.dropdown("config_file", "", [""] + sorted(config_entries), "configuration file")

In [0]:
# load configs using parameter values
config_file = dbutils.widgets.get("config_file")
config_file_path = f"{CONFIG_BASE_PATH}/{config_file}"

config = Commonlib.get_object_config(config_file_path)

In [0]:
# Instantiate medallion object
medallion = Medallion(config)

In [0]:
SOURCE_TABLES = {
    # silver source tables
    "us_cnty": MTU.get_fully_qualified_table_name("external", "static", "us_cnty"),
    "wslr_cust_crctd_addr": MTU.get_fully_qualified_table_name("oracle", "bud", "wslr_cust_crctd_addr"),
}

In [0]:
us_cnty_df = spark.read.table(SOURCE_TABLES["us_cnty"]).select("fips_st_cd", "fips_cnty_cd", "st_cd").alias("us_cnty")

In [0]:
wslr_cust_crctd_addr_df = (
    spark.read.table(SOURCE_TABLES["wslr_cust_crctd_addr"])
    .select(
        "wslr_cust_nbr",
        "congress_dist_nbr",
        "fips_state_cd",
        "fips_county_cd",
        "msa_nbr",
        "census_block_nbr",
        "ace_geo_match_cd",
        "ace_status_cd",
        "crctd_nm",
        "crctd_line1_addr",
        "crctd_line2_addr",
        "crctd_line2_addr",
        "crctd_city_nm",
        "crctd_st_abrv",
        "crctd_zip_cd",
        "crctd_cnty_nm",
        "crctd_lngtd",
        "crctd_lattd",
        "lattd_lngtd_err_cd",
        "lattd_lngtd_cnfdenc_lvl_cd",
        "geocoding_rslt_cd",
    )
    .alias("wslr_cust_crctd_addr")
)

In [0]:
df = wslr_cust_crctd_addr_df.join(
    us_cnty_df,
    (wslr_cust_crctd_addr_df.fips_state_cd == us_cnty_df.fips_st_cd)
    & (wslr_cust_crctd_addr_df.fips_county_cd == us_cnty_df.fips_cnty_cd),
    how="left",
)

In [0]:
# to remove duplicate columns
df = df.select(
    wslr_cust_crctd_addr_df.wslr_cust_nbr,
    wslr_cust_crctd_addr_df.congress_dist_nbr,
    wslr_cust_crctd_addr_df.fips_state_cd,
    wslr_cust_crctd_addr_df.fips_county_cd,
    us_cnty_df.st_cd,
    wslr_cust_crctd_addr_df.msa_nbr,
    wslr_cust_crctd_addr_df.census_block_nbr,
    wslr_cust_crctd_addr_df.ace_geo_match_cd,
    wslr_cust_crctd_addr_df.ace_status_cd,
    wslr_cust_crctd_addr_df.crctd_nm,
    wslr_cust_crctd_addr_df.crctd_line1_addr,
    wslr_cust_crctd_addr_df.crctd_line2_addr,
    wslr_cust_crctd_addr_df.crctd_city_nm,
    wslr_cust_crctd_addr_df.crctd_st_abrv,
    wslr_cust_crctd_addr_df.crctd_zip_cd,
    wslr_cust_crctd_addr_df.crctd_cnty_nm,
    wslr_cust_crctd_addr_df.crctd_lngtd,
    wslr_cust_crctd_addr_df.crctd_lattd,
    wslr_cust_crctd_addr_df.lattd_lngtd_err_cd,
    wslr_cust_crctd_addr_df.lattd_lngtd_cnfdenc_lvl_cd,
    wslr_cust_crctd_addr_df.geocoding_rslt_cd,
)

medallion.df = df.withColumn("__created_tsp", F.lit(medallion.start_datetime).cast(TimestampType()))


In [0]:
# write gold table
try:
    medallion.write_to_delta(load_type="overwrite", layer="gold")
except Exception as e:
    medallion.logger.error(
        f"Error writing wholesaler customer corrected address table to gold layer. Error message: {e}"
    )
    raise

In [0]:
# publish to external stage - snowflake
try:
    if config.get("publish_to_sf", False):
        pub = SnowflakeExternalTable(medallion.catalog, medallion.schema, medallion.name)
        pub.publish_ext_table()

except Exception as e:
    medallion.logger.error(e)
    raise

In [0]:
# compare legacy and modern datasets
try:
    medallion.compare_legacy_and_modern(
        config["name"],
        config["comparison_check"]["legacy_query"],
        config["comparison_check"]["modern_query"],
        medallion.key_columns,
        config.get("comparison_check", {}).get("completeness_lower_bound"),
        config.get("comparison_check", {}).get("equivalency_lower_bound"),
        config.get("comparison_check", {}).get("col_exclusions"),
        config.get("comparison_check", {}).get("auto_resolve_col_diffs"),
        config.get("comparison_check", {}).get("treat_nulls"),
    )
except KeyError as e:
    medallion.logger.warning(
        "Skipping comparison of legacy and modern because the required config is missing or invalid."
    )